# ECG Heartbeat Classification Project
## Phase 4 — Machine Learning Model Training

This notebook loads the extracted feature dataset from Phase 3
and trains machine learning models to classify heartbeats as
Normal or Abnormal. Two classifiers are trained and compared —
Random Forest and Support Vector Machine. Models are evaluated
using accuracy, precision, recall and F1-score to handle the
class imbalance in the dataset.

### Import libraries and load feature dataset
Loading the feature table saved in Phase 3 containing 102,382
heartbeat segments with 13 extracted features each.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score,
                             confusion_matrix, classification_report)
from sklearn.preprocessing import StandardScaler
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

# Load the feature dataset saved in Phase 3
df = pd.read_csv('../data/ecg_features.csv')

print("Dataset loaded successfully!")
print(f"\nShape: {df.shape}")
print(f"Rows (beats):    {df.shape[0]:,}")
print(f"Columns:         {df.shape[1]}")
print(f"\nFirst 5 rows:")
print(df.head())
print(f"\nClass distribution:")
print(f"  Normal   (0): {(df['label']==0).sum():,} beats ({(df['label']==0).sum()/len(df)*100:.1f}%)")
print(f"  Abnormal (1): {(df['label']==1).sum():,} beats ({(df['label']==1).sum()/len(df)*100:.1f}%)")

Dataset loaded successfully!

Shape: (102382, 14)
Rows (beats):    102,382
Columns:         14

First 5 rows:
       mean       std    max    min  range  r_peak_amplitude  skewness  \
0 -0.309792  0.169131  0.940 -0.535  1.475             0.940  5.099975   
1 -0.332389  0.153550  0.960 -0.570  1.530             0.885  5.974568   
2 -0.332958  0.145797  0.860 -0.645  1.505             0.810  5.654252   
3 -0.331958  0.138210  0.820 -0.565  1.385             0.820  5.651994   
4 -0.325750  0.146924  0.885 -0.545  1.430             0.885  5.526280   

    kurtosis  qrs_width  rr_interval  mean_first_half  mean_second_half  \
0  30.913988   0.019444     0.811111        -0.273333         -0.346250   
1  41.714788   0.016667     0.788889        -0.315139         -0.349639   
2  39.148715   0.016667     0.791667        -0.335917         -0.330000   
3  39.482946   0.013889     0.788889        -0.309944         -0.353972   
4  37.605972   0.016667     0.816667        -0.304556         -0.34694

### Splitting data into training and testing sets
The dataset is split 80/20 into training and testing sets.
The model learns from the training set and is evaluated on
the testing set which it has never seen before — proving
genuine learning rather than memorisation.

In [2]:
# Separate features (X) from labels (y)
# X contains all 13 feature columns
# y contains just the label column (0=Normal, 1=Abnormal)
X = df.drop('label', axis=1)
y = df['label']

print(f"Features (X) shape: {X.shape}")
print(f"Labels (y) shape:   {y.shape}")
print(f"Feature names: {list(X.columns)}")

# Split into 80% training and 20% testing
# random_state=42 ensures we get the same split every time we run
# stratify=y ensures both sets have the same class distribution
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"\nTraining set:")
print(f"  X_train shape: {X_train.shape}")
print(f"  Normal beats:   {(y_train==0).sum():,} ({(y_train==0).sum()/len(y_train)*100:.1f}%)")
print(f"  Abnormal beats: {(y_train==1).sum():,} ({(y_train==1).sum()/len(y_train)*100:.1f}%)")

print(f"\nTesting set:")
print(f"  X_test shape:  {X_test.shape}")
print(f"  Normal beats:   {(y_test==0).sum():,} ({(y_test==0).sum()/len(y_test)*100:.1f}%)")
print(f"  Abnormal beats: {(y_test==1).sum():,} ({(y_test==1).sum()/len(y_test)*100:.1f}%)")

# Scale the features — important for SVM
# StandardScaler converts all features to same scale (mean=0, std=1)
# This prevents features with large values dominating smaller ones
scaler = StandardScaler()

# Fit scaler on training data only — never on test data
# This prevents data leakage
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"\nFeature scaling complete!")
print(f"  Training mean after scaling: {X_train_scaled.mean():.4f} (should be ~0)")
print(f"  Training std after scaling:  {X_train_scaled.std():.4f} (should be ~1)")

Features (X) shape: (102382, 13)
Labels (y) shape:   (102382,)
Feature names: ['mean', 'std', 'max', 'min', 'range', 'r_peak_amplitude', 'skewness', 'kurtosis', 'qrs_width', 'rr_interval', 'mean_first_half', 'mean_second_half', 'half_difference']

Training set:
  X_train shape: (81905, 13)
  Normal beats:   60,008 (73.3%)
  Abnormal beats: 21,897 (26.7%)

Testing set:
  X_test shape:  (20477, 13)
  Normal beats:   15,003 (73.3%)
  Abnormal beats: 5,474 (26.7%)

Feature scaling complete!
  Training mean after scaling: -0.0000 (should be ~0)
  Training std after scaling:  1.0000 (should be ~1)


### Training Random Forest Classifier
Random Forest builds hundreds of decision trees on random subsets
of the training data and combines their predictions by majority
voting. class_weight='balanced' automatically compensates for the
73/27 class imbalance by giving more importance to Abnormal beats
during training.

In [4]:
print("Training Random Forest classifier...")

# Create Random Forest with 100 trees
# class_weight='balanced' handles the 73/27 class imbalance
# random_state=42 ensures reproducible results
rf_model = RandomForestClassifier(
    n_estimators=100,        # number of trees in the forest
    class_weight='balanced', # handle class imbalance
    random_state=42,         # reproducibility
    n_jobs=-1                # use all CPU cores for faster training
)

# Train the model on training data
# The model looks at 81,905 beats and learns the patterns
rf_model.fit(X_train, y_train)

print("Training complete!")

# Make predictions on the test set
# The model has never seen these 20,477 beats before
rf_predictions = rf_model.predict(X_test)

# Calculate evaluation metrics
rf_accuracy  = accuracy_score(y_test, rf_predictions)
rf_precision = precision_score(y_test, rf_predictions)
rf_recall    = recall_score(y_test, rf_predictions)
rf_f1        = f1_score(y_test, rf_predictions)

print(f"\nRandom Forest Results:")
print(f"{'─'*40}")
print(f"  Accuracy:  {rf_accuracy*100:.2f}%")
print(f"  Precision: {rf_precision*100:.2f}%")
print(f"  Recall:    {rf_recall*100:.2f}%")
print(f"  F1 Score:  {rf_f1*100:.2f}%")

# Full classification report
print(f"\nDetailed Classification Report:")
print(classification_report(y_test, rf_predictions,
      target_names=['Normal', 'Abnormal']))

Training Random Forest classifier...
Training complete!

Random Forest Results:
────────────────────────────────────────
  Accuracy:  97.26%
  Precision: 97.52%
  Recall:    92.07%
  F1 Score:  94.72%

Detailed Classification Report:
              precision    recall  f1-score   support

      Normal       0.97      0.99      0.98     15003
    Abnormal       0.98      0.92      0.95      5474

    accuracy                           0.97     20477
   macro avg       0.97      0.96      0.96     20477
weighted avg       0.97      0.97      0.97     20477



### Random Forest results

The Random Forest classifier achieved strong performance on the
test set of 20,477 unseen heartbeats:

- Accuracy of 97.26% — correctly classified the vast majority
- Precision of 97.52% — very few false abnormal predictions
- Recall of 92.07% — catches 92% of all abnormal beats
- F1 Score of 94.72% — strong balance between precision and recall

Normal beats show near perfect recall (0.99) while Abnormal beats
show slightly lower recall (0.92) — consistent with the EDA finding
that Atrial Premature beats are visually similar to Normal beats
making them harder to classify correctly.